<a href="https://colab.research.google.com/github/SathyaPrakashD/ml-pipeline-fundamentals/blob/main/08_fetch_california_housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
# ============================================================
# CHECKPOINT 1 — Load & Inspect Data
# ============================================================

from sklearn.datasets import fetch_california_housing
import pandas as pd

# Step 1 — Load the dataset
data = fetch_california_housing()
#print(data.feature_names)
# Step 2 — Wrap features into a DataFrame
# Hint: data.feature_names gives you column names
df = pd.DataFrame(data.data, columns=data.feature_names)





# Step 3 — Add target as number
df['target'] = data.target
# df['target_name'] = data.target_names[data.target] # This line caused the error

print("Shape:", df.shape)
print("Classes:", df['target'].value_counts())
# print("Class counts:\n", df['target_name'].value_counts()) # This line is also problematic without target_name
print("\nFirst 5 rows:")
print(df.head())

Shape: (20640, 9)
Classes: target
5.00001    965
1.37500    122
1.62500    117
1.12500    103
1.87500     93
          ... 
0.34200      1
0.46200      1
3.52000      1
3.07900      1
3.85200      1
Name: count, Length: 3842, dtype: int64

First 5 rows:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  target  
0    -122.23   4.526  
1    -122.22   3.585  
2    -122.24   3.521  
3    -122.25   3.413  
4    -122.25   3.422  


In [21]:

# ============================================================
# CHECKPOINT 2 — Split first, then Scale (correct order)
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Step 1 — Separate features and target
X = df.drop(columns=['target'])
y = df['target']

# Step 2 — Split BEFORE scaling
# test_size=0.2 means 20% test, 80% train
# random_state fixes the random split so results are reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

# Step 3 — Fit scaler on TRAIN only
scaler = StandardScaler()
scaler.fit(X_train)             # learns mean & std from train only

# Step 4 — Transform both using train's stats
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)



# Step 5 — Verify scaling worked
X_train_df_before = pd.DataFrame(X_train, columns=X.columns)
print("\nBefore scaling (train set):")
print(X_train_df_before.head())

X_train_df = pd.DataFrame(X_train_scaled, columns=X.columns)
print("\nAFTER scaling (train set):")
print(X_train_df.head())

Train size: 16512
Test size: 4128

Before scaling (train set):
       MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
14196  3.2596      33.0  5.017657   1.006421      2300.0  3.691814     32.71   
8267   3.8125      49.0  4.473545   1.041005      1314.0  1.738095     33.77   
17445  4.1563       4.0  5.645833   0.985119       915.0  2.723214     34.66   
14265  1.9425      36.0  4.002817   1.033803      1418.0  3.994366     32.69   
2271   3.5542      43.0  6.268421   1.134211       874.0  2.300000     36.78   

       Longitude  
14196    -117.03  
8267     -118.16  
17445    -120.48  
14265    -117.11  
2271     -119.80  

AFTER scaling (train set):
     MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0 -0.326196  0.348490 -0.174916  -0.208365    0.768276  0.051376 -1.372811   
1 -0.035843  1.618118 -0.402835  -0.128530   -0.098901 -0.117362 -0.876696   
2  0.144701 -1.952710  0.088216  -0.257538   -0.449818 -0.032280 -0.460146   
3 

In [22]:
# ============================================================
# CHECKPOINT 3 — Train Linear Regression
# ============================================================

from sklearn.linear_model import LinearRegression # Changed from LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score # Added for regression metrics
import numpy as np
from sklearn.metrics import mean_absolute_error

# Step 1 — Create the model
model = LinearRegression()   # Changed from LogisticRegression

# Step 2 — Train it (only on train data)
model.fit(X_train_scaled, y_train)   # Model learns patterns and relationships

# Step 3 — Predict on test set
y_pred = model.predict(X_test_scaled)   #

# Step 4 — Score it (using regression metrics)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"MAE: {mae:.4f}")
print(f"Mean Squared Error: {mse:.4f}")
print(f"R-squared: {r2:.4f}")
print(f"rmse : {rmse:.4f}")


# Step 5 — Inspect one prediction
print("\nFirst test sample — true label   :", y_test.values[0])
print("First test sample — predicted    :", y_pred[0])
# For Linear Regression, there are no probabilities, only direct predictions
# print("First test sample — probabilities:", model.predict_proba(X_test_scaled)[0].round(3))

MAE: 0.5332
Mean Squared Error: 0.5559
R-squared: 0.5758
rmse : 0.7456

First test sample — true label   : 0.477
First test sample — predicted    : 0.7191228416019144


In [19]:
# ============================================================
# CHECKPOINT 4 — Cross Validation
# ============================================================

from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import numpy as np # Import numpy for mean and std

# Step 1 — Build a pipeline (scaler + model in one object)
# This ensures scaling happens correctly inside each fold
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

# Step 2 — Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Step 3 — Make a prediction with the fitted pipeline
import pandas as pd

custom_district = pd.DataFrame([[3.5, 30.0, 5.5, 1.0, 800.0, 2.5, 37.85, -122.25]],
    columns=['MedInc','HouseAge','AveRooms','AveBedrms',
             'Population','AveOccup','Latitude','Longitude'])

predicted_price = pipeline.predict(custom_district)[0]
print(f"Predicted house price for custom district: ${predicted_price * 100000:,.0f}")

# Step 4 — Perform cross-validation (using the pipeline for robustness)
# Using 'neg_mean_squared_error' as scoring for regression models
cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='neg_mean_squared_error')
rmse_scores = np.sqrt(-cv_scores) # Convert negative MSE to RMSE

print(f"\nCross-validation RMSE scores: {rmse_scores.round(4)}")
print(f"Mean CV RMSE: {rmse_scores.mean():.4f}")
print(f"Std CV RMSE: {rmse_scores.std():.4f}")

Predicted house price for custom district: $206,490

Cross-validation RMSE scores: [0.6963 0.789  0.8039 0.737  0.7033]
Mean CV RMSE: 0.7459
Std CV RMSE: 0.0437


In [17]:
print(f"Raw prediction: {predicted_price}")
print(f"Predicted house price: ${predicted_price * 100000:,.0f}")

Raw prediction: 2.064904556338354
Predicted house price: $206,490
